In [1]:
import os

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.system("copy kaggle.json %USERPROFILE%\\.kaggle\\")

0

In [2]:
!pip install --upgrade kaggle

In [3]:
!kaggle --version

Kaggle CLI 2.0.1


In [4]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
import zipfile

with zipfile.ZipFile("utkface-new.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [6]:
import os

print(len(os.listdir("dataset")))

3


In [7]:
import os
print(os.listdir("dataset"))

['crop_part1', 'UTKFace', 'utkface_aligned_cropped']


In [8]:
print(len(os.listdir("dataset/UTKFace")))

23708


In [9]:
data_path = "dataset/UTKFace"

In [10]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [11]:
data_path = "dataset/UTKFace"
images, labels = [], []

for img_name in os.listdir(data_path):
    try:
        gender = int(img_name.split("_")[1])
        img_path = os.path.join(data_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (64, 64))
        img = img / 255.0
        images.append(img)
        labels.append(gender)
    except:
        continue

In [12]:
X = np.array(images)
y = np.array(labels)
print(X.size)
print(y.size)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

291323904
23708


In [13]:
import tensorflow as tf
model = tf.keras.models.Sequential()
# Conv Layer 1
model.add(tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)))
model.add(tf.keras.layers.MaxPooling2D(2,2))

# Conv Layer 2
model.add(tf.keras.layers.Conv2D(64, (3,3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(2,2))

# Conv Layer 3
model.add(tf.keras.layers.Conv2D(128, (3,3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(2,2))

# Flatten
model.add(tf.keras.layers.Flatten())

# Dense Layers
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))  # Binary output

C:\Users\singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
460/593 ━━━━━━━━━━━━━━━━━━━━ 16s 126ms/step - accuracy: 0.6854 - loss: 0.5602

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title("Accuracy")
plt.show()